In [33]:
import numpy as np
import dpluspy 
import pandas

In [34]:
bands = pandas.read_csv("../cytoBand.txt", sep="\s+", names=["chrom", "start", "end", "band", "stain"])
seqlens = pandas.read_csv("../hg19.genome", sep="\s+", names=["chrom", "seqlen"])

<>:1: DeprecationWarning: invalid escape sequence '\s'


In [35]:
def find_breakpoints(regions, target_size):
    """
    Find points that divide a mask, `regions`, into chunks of approximately
    `target_size` sites each.
    """
    tally = 0
    last_end = 0
    breakpoints = []
    for start, end in regions:
        length = end - start 
        if tally > target_size:
            print(tally)
            breakpoints.append(last_end)
            tally = length 
        else: 
            tally += length 
        last_end = end
    if tally >= target_size / 2:
        breakpoints.append(last_end)
    return breakpoints

In [36]:
def set_up_intervals(regions, sites_per_window, cen=None):
    """
    
    :param int cen: Position corresponding to the center of the centromere
        (default None)
    """
    if cen is not None: 
        regions0 = regions[regions[:, 1] < cen]
        regions1 = regions[regions[:, 1] >= cen]

        intervals = []

        breakpoints0 = find_breakpoints(regions0, sites_per_window)
        # Account for 1-indexing
        breakpoints0 = np.array([0] + breakpoints0) + 1
        for ii in range(len(breakpoints0) - 1):
            interval = np.array(
                [breakpoints0[ii], breakpoints0[ii + 1], breakpoints0[-1]])
            intervals.append(interval)

        breakpoints1 = find_breakpoints(regions1, sites_per_window)
        # Account for 1-indexing
        breakpoints1 = np.array([cen] + breakpoints1) + 1
        for ii in range(len(breakpoints1) - 1):
            interval = np.array(
                [breakpoints1[ii], breakpoints1[ii + 1], breakpoints1[-1]])
            intervals.append(interval)

    else:
        breakpoints = find_breakpoints(regions, sites_per_window)
        # Account for 1-indexing
        breakpoints = np.array([0] + breakpoints) + 1
        intervals = []
        for ii in range(len(breakpoints) - 1):
            interval = np.array(
                [breakpoints[ii], breakpoints[ii + 1], breakpoints[-1]])
            intervals.append(interval)
    return intervals

In [37]:
mask_fname = "../bed_files/filterbed_intergenic/mask_chr20.bed.gz"
regions, _ = dpluspy.utils._read_bed_file(mask_fname)
set_up_intervals(regions, 10000000, cen=30000000)

10000077
10000005


[array([       1, 21353092, 21353092]), array([30000001, 53582694, 53582694])]

In [38]:
target_size = 1300000
for ii in range(1, 23):
    mask_fname = f"../bed_files/filterbed_1e-4M_exon_buffer/mask_chr{ii}.bed.gz"
    regions, _ = dpluspy.utils._read_bed_file(mask_fname)
    if ii in (13, 14, 15, 21, 22):
        intervals = set_up_intervals(regions, target_size)
    else:        
        centromere = bands[(bands["chrom"] == f"chr{ii}") 
                           & (bands["stain"] == "acen")]
        cen = (np.min(centromere["start"]) + np.max(centromere["end"])) / 2
        intervals = set_up_intervals(regions, target_size, cen=cen)
    np.savetxt(f"1e-4_exon_buffer_1.3Mb/intervals_chr{ii}.txt", intervals)


1300055
1300045
1300002
1300010
1300035
1300003
1300001
1300045
1300055
1300017
1300001
1300084
1300028
1300006
1300006
1300008
1300033
1300017
1300003
1300005
1300015
1300098
1300066
1300042
1300009
1300021
1300019
1300014
1300034
1300024
1300061
1300028
1300028
1300112
1300001
1300064
1300041
1300041
1300071
1300108
1300008
1300008
1300006
1300021
1300037
1300005
1300005
1300015
1300001
1300004
1300045
1300019
1300036
1300024
1300001
1300031
1300025
1300043
1300046
1300003
1300016
1300001
1300001
1300048
1300045
1300026
1300013
1300035
1300134
1300010
1300017
1300007
1300054
1300012
1300020
1300056
1300009
1300141
1300013
1300043
1300001
1300010
1300030
1300086
1300002
1300001
1300002
1300008
1300064
1300005
1300052
1300013
1300074
1300017
1300025
1300054
1300047
1300002
1300011
1300001
1300002
1300003
1300004
1300023
1300023
1300022
1300008
1300002
1300003
1300055
1300077
1300019
1300019
1300011
1300034
1300002
1300016
1300043
1300013
1300098
1300043
1300005
1300002
1300004
1300003
